> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 5 · Notebook 04 — Volatility estimators and adaptive trend

**Sessions:** S6 (Direction group) · S7 (Volatility & momentum groups) · [Lesson plan](../../docs/lessons/PART_05_ANALYTICS_LIBRARY.md) · graded labs in [`labs/part05/`](../../labs/part05/)

**You will:**
1. Estimate volatility from the high–low range (Parkinson, Garman–Klass).
2. Measure how much more precise range estimators are, on bars with a known σ.
3. Write Kaufman's efficiency ratio, the engine of the adaptive moving average.
4. See KAMA follow a trend and sit still in chop.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure, so you always know which effects are real and which are luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p5lib.py is in notebooks/part05/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p5lib as p

p.use_course_style()

## 1. Bars with a known volatility

To test an estimator you need the right answer. `p.brownian_bars` simulates each day as 390 one-minute steps of a random walk with a daily σ of **1%**, and records the open, high, low and close. No drift, no overnight gap: the textbook world these estimators assume.

In [ ]:
bars = p.brownian_bars(500, sigma=0.01, seed=0)
o, h, l, c = (bars[k].to_numpy() for k in ("open", "high", "low", "close"))
print(f"close-to-close estimate over 500 days: {p.close_to_close_vol(c):.4%} (true 1.0000%)")
bars.head()

## 2. Range estimators

The close-to-close estimator uses one number per day. The range uses the path inside the day:

* **Parkinson:** `σ² = mean(ln(H/L)²) / (4 ln 2)`
* **Garman–Klass:** `σ² = mean(½ ln(H/L)² − (2 ln 2 − 1) ln(C/O)²)`

Return the daily σ (the square root).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def parkinson_vol(h, l):
    return float(np.sqrt(np.mean(np.log(h / l) ** 2) / (4 * np.log(2))))

def garman_klass_vol(o, h, l, c):
    return float(np.sqrt(np.mean(0.5 * np.log(h / l) ** 2 - (2 * np.log(2) - 1) * np.log(c / o) ** 2)))

mine = [parkinson_vol(h, l), garman_klass_vol(o, h, l, c)]
mine = p.check("range estimators", mine, [p.parkinson_vol(h, l), p.garman_klass_vol(o, h, l, c)])
[f"{x:.4%}" for x in mine]

Both come out slightly **below** 1%: we only see 390 prices a day, so the observed high and low miss the true extremes between them. Real data has the same bias, plus gaps and drift (Yang–Zhang, in the lab, handles those). The prize is **precision**: over short windows the range estimators wobble much less.

In [ ]:
est = {"close-to-close": [], "Parkinson": [], "Garman–Klass": []}
for s in range(300):
    b = p.brownian_bars(20, sigma=0.01, seed=100 + s)
    o_, h_, l_, c_ = (b[k].to_numpy() for k in ("open", "high", "low", "close"))
    est["close-to-close"].append(np.sqrt(np.mean(np.log(c_ / o_) ** 2)))   # one day's return is open→close here
    est["Parkinson"].append(p.parkinson_vol(h_, l_))
    est["Garman–Klass"].append(p.garman_klass_vol(o_, h_, l_, c_))
fig, ax = plt.subplots()
for k, vals in est.items():
    ax.hist(np.array(vals) * 100, bins=30, alpha=0.6, label=k)
ax.axvline(1.0, color="black", lw=1)
ax.set(xlabel="estimated daily σ, % (20-day windows)", ylabel="windows", title="Same data, different precision"); ax.legend(); plt.show()
sd = {k: np.std(v) for k, v in est.items()}
for k in ("Parkinson", "Garman–Klass"):
    print(f"{k}: {sd['close-to-close'] ** 2 / sd[k] ** 2:.1f}× as efficient as close-to-close (variance ratio)")

## 3. The efficiency ratio

Kaufman's **efficiency ratio** asks how much of the path was progress: the net move over `n` bars divided by the sum of the absolute bar-to-bar moves over the same bars. 1 is a straight line; near 0 is chop. `ER[t] = |C[t] − C[t−n]| / Σ|ΔC|` over those `n` changes (0 when the sum is 0), NaN for `t < n`.

In [ ]:
x = p.trend_then_chop()
plt.plot(x); plt.title("200 bars of trend, then 200 bars of chop"); plt.show()

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

Hint: `pd.Series(np.abs(np.diff(close))).rolling(n).sum()` gives the path length ending at each change; the one that matches `change[0]` (bars 0…n) is at position `n − 1`.

In [ ]:
def efficiency_ratio(close, n=10):
    out = np.full(close.shape, np.nan)
    change = np.abs(close[n:] - close[:-n])                           # net move, aligned with bars n, n+1, …
    path = pd.Series(np.abs(np.diff(close))).rolling(n).sum().to_numpy()[n - 1:]
    out[n:] = np.divide(change, path, out=np.zeros_like(change), where=path > 0)
    return out

mine = p.attempt(efficiency_ratio, x, 10)
mine = p.check("efficiency_ratio", mine, p.efficiency_ratio(x, 10))
print(f"mean ER: trend {np.nanmean(mine[:200]):.2f}, chop {np.nanmean(mine[200:]):.2f}")

## 4. KAMA: an average that speeds up in trends

KAMA uses the ER to set its smoothing constant each bar: between a fast EMA (2) in a clean trend and a slow EMA (30) in chop. Count how often each line changes direction in the chop half: every change is a potential whipsaw trade.

In [ ]:
k, e = p.kama(x, 10), p.ema(x, 10)
fig, ax = plt.subplots()
ax.plot(x, lw=0.8, color="#b5b4ad", label="price"); ax.plot(e, label="EMA 10"); ax.plot(k, label="KAMA 10")
ax.axvline(200, color="black", lw=0.8); ax.set_title("KAMA follows the trend and goes flat in the chop"); ax.legend(); plt.show()
flips = {name: int(np.sum(np.diff(np.sign(np.diff(series[200:]))) != 0)) for name, series in [("EMA 10", e), ("KAMA 10", k)]}
print("direction changes in the chop half:", flips)

## Wrap-up

* Range-based volatility is several times more efficient than close-to-close: shorter windows, same precision.
* Test estimators on data with a known answer before trusting them on real data.
* Adaptive averages trade some lag in trends for fewer direction changes (whipsaws) in chop.
* Graded version: `labs/part05/week18_groups` (five estimators including Yang–Zhang, KAMA and ADX against TA-Lib, SuperTrend).